In [1]:
from pyspark.sql import functions as F

# ============================================================
# AirOps 360 - Week 4 Task 5
# Bronze BTS -> standardized Silver flights v0.1
#
# NOT IN SCOPE TODAY:
# flight_key, dedupe, quarantine, weather join, Gold, Power BI
# ============================================================

SOURCE_TABLE = "lh_airops_bronze.brz_bts_flights"
TARGET_TABLE = "slv_flights"

BATCH_KEY = "bts_reporting_carrier_ontime|2026|04"
EXPECTED_ROWS = 597_919
SILVER_VERSION = "0.1"

LINEAGE_COLS = [
    "_bronze_run_id",
    "_bronze_load_id",
    "_bronze_batch_key",
    "_bronze_contract_version",
    "_bronze_source_name",
    "_bronze_source_file_name",
    "_bronze_source_hash",
    "_bronze_ingested_at_utc",
    "_bronze_load_year",
    "_bronze_load_month",
]


# ============================================================
# Helper functions
# ============================================================

def clean_string(name):
    """
    Trim strings and convert blank / textual null values to Spark NULL.
    """
    s = F.trim(F.col(name).cast("string"))

    return (
        F.when(
            F.col(name).isNull(),
            F.lit(None).cast("string")
        )
        .when(
            F.lower(s).isin("", "nan", "null", "none"),
            F.lit(None).cast("string")
        )
        .otherwise(s)
    )


def clean_double(name):
    """
    Convert numeric field to double while standardizing NaN -> NULL.
    """
    d = F.col(name).cast("double")

    return (
        F.when(
            F.col(name).isNull() | F.isnan(d),
            F.lit(None).cast("double")
        )
        .otherwise(d)
    )


def clean_int(name):
    """
    Convert numeric field to integer while standardizing NaN -> NULL.
    """
    d = F.col(name).cast("double")

    return (
        F.when(
            F.col(name).isNull() | F.isnan(d),
            F.lit(None).cast("int")
        )
        .otherwise(d.cast("int"))
    )


def clean_bool01(name):
    """
    Standardize BTS numeric 0/1 fields into true/false.
    Anything else becomes NULL rather than being silently coerced.
    """
    d = F.col(name).cast("double")

    return (
        F.when(
            F.col(name).isNull() | F.isnan(d),
            F.lit(None).cast("boolean")
        )
        .when(d == 1.0, F.lit(True))
        .when(d == 0.0, F.lit(False))
        .otherwise(F.lit(None).cast("boolean"))
    )


def hhmm_string(name):
    """
    Convert BTS HHMM numeric representation:
        5    -> 00:05
        730  -> 07:30
        1425 -> 14:25
        2400 -> 00:00

    Full cross-midnight timestamps are intentionally NOT created today.
    """
    d = F.col(name).cast("double")

    x = (
        F.when(
            F.col(name).isNull() | F.isnan(d),
            F.lit(None).cast("int")
        )
        .otherwise(d.cast("int"))
    )

    valid = (
        (x == 2400)
        |
        (
            (x >= 0)
            & (x <= 2359)
            & ((x % 100) < 60)
        )
    )

    normalized = F.when(x == 2400, 0).otherwise(x)

    hh = F.floor(normalized / 100).cast("int")
    mm = (normalized % 100).cast("int")

    return (
        F.when(
            x.isNull(),
            F.lit(None).cast("string")
        )
        .when(
            valid,
            F.format_string("%02d:%02d", hh, mm)
        )
        .otherwise(
            F.lit(None).cast("string")
        )
    )


# ============================================================
# 1. READ VERIFIED APRIL BRONZE BATCH
# ============================================================

bronze = (
    spark.table(SOURCE_TABLE)
         .filter(F.col("_bronze_batch_key") == BATCH_KEY)
)

bronze_rows = bronze.count()

print(f"Bronze rows: {bronze_rows:,}")

assert bronze_rows == EXPECTED_ROWS, (
    f"STOP: expected {EXPECTED_ROWS:,} Bronze rows, "
    f"but found {bronze_rows:,}"
)

missing_lineage = [
    c for c in LINEAGE_COLS
    if c not in bronze.columns
]

assert not missing_lineage, (
    f"STOP: missing Bronze lineage columns: {missing_lineage}"
)


# ============================================================
# 2. VALIDATE HHMM SOURCE VALUES BEFORE TRANSFORMING
# ============================================================

TIME_SOURCE_COLS = [
    "CRSDepTime",
    "DepTime",
    "WheelsOff",
    "CRSArrTime",
    "ArrTime",
    "WheelsOn",
]

invalid_time_expressions = []

for c in TIME_SOURCE_COLS:

    d = F.col(c).cast("double")
    x = d.cast("int")

    valid_numeric = (
        d.isNotNull()
        & (~F.isnan(d))
    )

    valid_hhmm = (
        valid_numeric
        &
        (
            (x == 2400)
            |
            (
                (x >= 0)
                & (x <= 2359)
                & ((x % 100) < 60)
            )
        )
    )

    invalid = F.when(
        F.col(c).isNotNull() & (~valid_hhmm),
        1
    ).otherwise(0)

    invalid_time_expressions.append(
        F.sum(invalid).alias(c)
    )


invalid_times = (
    bronze
    .agg(*invalid_time_expressions)
    .first()
    .asDict()
)

print("Invalid HHMM values:")
print(invalid_times)

if any(v > 0 for v in invalid_times.values()):
    raise ValueError(
        "STOP: malformed HHMM values found. "
        "Do not silently publish malformed time values."
    )


# ============================================================
# 3. BUILD SILVER STANDARDIZED DATASET
# ============================================================

silver = bronze.select(

    # ------------------------
    # Calendar
    # ------------------------

    clean_int("Year").alias("year"),
    clean_int("Quarter").alias("quarter"),
    clean_int("Month").alias("month"),
    clean_int("DayofMonth").alias("day_of_month"),
    clean_int("DayOfWeek").alias("day_of_week"),

    F.to_date(
        F.col("FlightDate")
    ).alias("flight_date"),


    # ------------------------
    # Carrier / flight
    # ------------------------

    clean_string(
        "Reporting_Airline"
    ).alias("reporting_airline"),

    clean_int(
        "DOT_ID_Reporting_Airline"
    ).alias("airline_dot_id"),

    clean_string(
        "IATA_CODE_Reporting_Airline"
    ).alias("airline_iata_code"),

    clean_string(
        "Tail_Number"
    ).alias("tail_number"),

    clean_int(
        "Flight_Number_Reporting_Airline"
    ).alias("flight_number"),


    # ------------------------
    # Origin
    # ------------------------

    clean_int(
        "OriginAirportID"
    ).alias("origin_airport_id"),

    clean_string(
        "Origin"
    ).alias("origin"),

    clean_string(
        "OriginCityName"
    ).alias("origin_city_name"),

    clean_string(
        "OriginState"
    ).alias("origin_state"),


    # ------------------------
    # Destination
    # ------------------------

    clean_int(
        "DestAirportID"
    ).alias("dest_airport_id"),

    clean_string(
        "Dest"
    ).alias("dest"),

    clean_string(
        "DestCityName"
    ).alias("dest_city_name"),

    clean_string(
        "DestState"
    ).alias("dest_state"),


    # ------------------------
    # Departure
    # ------------------------

    clean_int(
        "CRSDepTime"
    ).alias("crs_dep_time_hhmm"),

    hhmm_string(
        "CRSDepTime"
    ).alias("crs_dep_time_local"),

    clean_int(
        "DepTime"
    ).alias("dep_time_hhmm"),

    hhmm_string(
        "DepTime"
    ).alias("dep_time_local"),

    clean_double(
        "DepDelay"
    ).alias("dep_delay_minutes_signed"),

    clean_double(
        "DepDelayMinutes"
    ).alias("dep_delay_minutes"),

    clean_bool01(
        "DepDel15"
    ).alias("dep_del15"),

    clean_double(
        "TaxiOut"
    ).alias("taxi_out_minutes"),

    clean_int(
        "WheelsOff"
    ).alias("wheels_off_hhmm"),

    hhmm_string(
        "WheelsOff"
    ).alias("wheels_off_local"),


    # ------------------------
    # Arrival
    # ------------------------

    clean_int(
        "CRSArrTime"
    ).alias("crs_arr_time_hhmm"),

    hhmm_string(
        "CRSArrTime"
    ).alias("crs_arr_time_local"),

    clean_int(
        "ArrTime"
    ).alias("arr_time_hhmm"),

    hhmm_string(
        "ArrTime"
    ).alias("arr_time_local"),

    clean_double(
        "ArrDelay"
    ).alias("arr_delay_minutes_signed"),

    clean_double(
        "ArrDelayMinutes"
    ).alias("arr_delay_minutes"),

    clean_bool01(
        "ArrDel15"
    ).alias("arr_del15"),

    clean_int(
        "WheelsOn"
    ).alias("wheels_on_hhmm"),

    hhmm_string(
        "WheelsOn"
    ).alias("wheels_on_local"),

    clean_double(
        "TaxiIn"
    ).alias("taxi_in_minutes"),


    # ------------------------
    # Flight status
    # ------------------------

    clean_bool01(
        "Cancelled"
    ).alias("cancelled"),

    clean_string(
        "CancellationCode"
    ).alias("cancellation_code"),

    clean_bool01(
        "Diverted"
    ).alias("diverted"),


    # ------------------------
    # Duration / distance
    # ------------------------

    clean_double(
        "CRSElapsedTime"
    ).alias("crs_elapsed_minutes"),

    clean_double(
        "ActualElapsedTime"
    ).alias("actual_elapsed_minutes"),

    clean_double(
        "AirTime"
    ).alias("air_time_minutes"),

    clean_int(
        "Flights"
    ).alias("flights"),

    clean_double(
        "Distance"
    ).alias("distance_miles"),


    # ------------------------
    # Delay causes
    # ------------------------

    clean_double(
        "CarrierDelay"
    ).alias("carrier_delay_minutes"),

    clean_double(
        "WeatherDelay"
    ).alias("weather_delay_minutes"),

    clean_double(
        "NASDelay"
    ).alias("nas_delay_minutes"),

    clean_double(
        "SecurityDelay"
    ).alias("security_delay_minutes"),

    clean_double(
        "LateAircraftDelay"
    ).alias("late_aircraft_delay_minutes"),


    # ------------------------
    # Bronze lineage
    # ------------------------

    *[
        F.col(c)
        for c in LINEAGE_COLS
    ],


    # ------------------------
    # Silver processing metadata
    # ------------------------

    F.lit(
        SILVER_VERSION
    ).alias("_silver_version"),

    F.current_timestamp(
    ).alias("_silver_transformed_at_utc"),
)


# ============================================================
# 4. PRE-WRITE VALIDATION
# ============================================================

silver_rows = silver.count()

assert silver_rows == bronze_rows, (
    f"STOP: standardization changed row count. "
    f"Bronze={bronze_rows:,}, Silver={silver_rows:,}"
)


flight_date_nulls = (
    silver
    .filter(F.col("flight_date").isNull())
    .count()
)

assert flight_date_nulls == 0, (
    f"STOP: FlightDate parse failures = {flight_date_nulls:,}"
)


outside_april = (
    silver
    .filter(
        (F.year("flight_date") != 2026)
        |
        (F.month("flight_date") != 4)
    )
    .count()
)

assert outside_april == 0, (
    f"STOP: rows outside April 2026 = {outside_april:,}"
)


lineage_null_counts = (
    silver
    .agg(
        *[
            F.sum(
                F.col(c).isNull().cast("int")
            ).alias(c)
            for c in LINEAGE_COLS
        ]
    )
    .first()
    .asDict()
)

lineage_null_total = sum(
    lineage_null_counts.values()
)

assert lineage_null_total == 0, (
    f"STOP: required Bronze lineage nulls = "
    f"{lineage_null_total:,}"
)


# Show new boolean domains

for c in [
    "cancelled",
    "diverted",
    "dep_del15",
    "arr_del15"
]:
    print(f"\n{c}:")
    silver.groupBy(c).count().orderBy(c).show()


print("\nSilver schema:")
silver.printSchema()


# ============================================================
# 5. WRITE SILVER DELTA TABLE
# ============================================================

(
    silver.write
          .format("delta")
          .mode("overwrite")
          .option("overwriteSchema", "true")
          .saveAsTable(TARGET_TABLE)
)


# ============================================================
# 6. POST-WRITE QA / EVIDENCE
# ============================================================

target = spark.table(TARGET_TABLE)

target_rows = target.count()

actual_types = dict(target.dtypes)

expected_types = {
    "flight_date": "date",
    "flight_number": "int",
    "crs_dep_time_hhmm": "int",
    "cancelled": "boolean",
    "diverted": "boolean",
    "dep_del15": "boolean",
    "arr_del15": "boolean",
    "distance_miles": "double",
}

type_failures = {
    col_name: (
        actual_types.get(col_name),
        expected_type
    )
    for col_name, expected_type
    in expected_types.items()
    if actual_types.get(col_name) != expected_type
}


print("\n" + "=" * 72)
print("AIR0PS 360 - TASK 5 SILVER STANDARDIZATION EVIDENCE")
print("=" * 72)

print(f"Source table:     {SOURCE_TABLE}")
print(f"Target table:     {TARGET_TABLE}")
print(f"Batch key:        {BATCH_KEY}")

print(f"Bronze rows:      {bronze_rows:,}")
print(f"Silver rows:      {target_rows:,}")

print(f"Silver columns:   {len(target.columns)}")

print(f"FlightDate nulls: {flight_date_nulls:,}")
print(f"Outside Apr 2026: {outside_april:,}")

print(f"Lineage nulls:    {lineage_null_total:,}")
print(f"Type failures:    {type_failures}")

print(f"Silver version:   {SILVER_VERSION}")

print("=" * 72)


assert target_rows == EXPECTED_ROWS, (
    f"STOP: target count {target_rows:,} "
    f"!= expected {EXPECTED_ROWS:,}"
)

assert not type_failures, (
    f"STOP: unexpected Silver types: {type_failures}"
)


print("\nTASK 5 STATUS: PASS")


# ============================================================
# 7. HUMAN-READABLE SAMPLE
# ============================================================

display(
    target.select(
        "flight_date",
        "reporting_airline",
        "flight_number",
        "origin",
        "dest",
        "crs_dep_time_hhmm",
        "crs_dep_time_local",
        "cancelled",
        "diverted",
        "_bronze_batch_key",
        "_silver_version",
    ).limit(20)
)

StatementMeta(, 0f4be612-324b-4c3b-bdc6-d526ee7d2c5e, 3, Finished, Available, Finished, False)

Bronze rows: 597,919
Invalid HHMM values:
{'CRSDepTime': 0, 'DepTime': 0, 'WheelsOff': 0, 'CRSArrTime': 0, 'ArrTime': 0, 'WheelsOn': 0}

cancelled:
+---------+------+
|cancelled| count|
+---------+------+
|    false|592517|
|     true|  5402|
+---------+------+


diverted:
+--------+------+
|diverted| count|
+--------+------+
|   false|596608|
|    true|  1311|
+--------+------+


dep_del15:
+---------+------+
|dep_del15| count|
+---------+------+
|     NULL|  5171|
|    false|473291|
|     true|119457|
+---------+------+


arr_del15:
+---------+------+
|arr_del15| count|
+---------+------+
|     NULL|  6713|
|    false|471789|
|     true|119417|
+---------+------+


Silver schema:
root
 |-- year: integer (nullable = true)
 |-- quarter: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day_of_month: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- flight_date: date (nullable = true)
 |-- reporting_airline: string (nullable = true)
 |-- airli

SynapseWidget(Synapse.DataFrame, 345d3466-8e3b-4d41-824a-131d48398635)